# 08 — Presentation Figures Generator

Generates all figures needed for the Data Preprocessing slides.

| Figure | Slide | What it shows |
|--------|-------|---------------|
| `blur_detection.png` | Slide 2 | Sharp vs blurry frame + Laplacian heatmaps |
| `phash_dedup.png` | Slide 2 | 3 near-identical frames + DCT hash grids |
| `severity_crops.png` | Slide 3 | Low/Medium/High pothole crops from raw images |
| `augmentation_grid.png` | Slide 4 | 5×5 weather augmentation grid |

---
**HOW TO USE THIS NOTEBOOK:**
1. Run **Section 0** (setup) once.
2. For each figure, go to that section, **set your image paths** in the `USER INPUT` cell.
3. Run all cells in that section.
4. The figure is saved to the project root as a `.png` file.

All input images should come from `data/raw/` — pick them from Windows Explorer and paste the path.

---
## Section 0 — Setup (run once)

In [1]:
import sys
sys.path.insert(0, '../src')

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path

plt.rcParams.update({'figure.dpi': 130, 'font.size': 10})

BASE_DIR    = Path('..').resolve()
RAW_DIR     = BASE_DIR / 'data' / 'raw'
OUTPUT_DIR  = BASE_DIR   # figures saved to project root

def load_rgb(path):
    img = cv2.imread(str(path))
    assert img is not None, f'Could not load image: {path}'
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def save_figure(fig, filename):
    out = OUTPUT_DIR / filename
    fig.savefig(str(out), dpi=150, bbox_inches='tight', facecolor='white')
    print(f'Saved → {out}')

print('Setup done.')
print('RAW_DIR    =', RAW_DIR)
print('Output dir =', OUTPUT_DIR)
print()
print('Raw subfolders:')
for d in sorted(RAW_DIR.iterdir()):
    if d.is_dir():
        print(f'  {d.name}/')

Setup done.
RAW_DIR    = D:\7 SEM\Computer Vision\Project\smart-road-assistant\data\raw
Output dir = D:\7 SEM\Computer Vision\Project\smart-road-assistant

Raw subfolders:
  potholes/
  traffic_lights/


---
## Figure 1 — Blur Detection (Laplacian Operator)
**Slide 2, Left half**

Shows one sharp frame (KEPT) and one blurry frame (REJECTED) side by side,
each with its greyscale and Laplacian heatmap.

**Where to pick images from:** `data/raw/dashcam_frames/`  
Pick one clearly sharp frame and one visibly blurry frame.

In [ ]:
# ┌─────────────────────────────────────────────────────────────────┐
# │  USER INPUT — set your two image paths below                    │
# │  Use forward slashes or raw strings. Examples:                  │
# │  r'..\data\raw\dashcam_frames\clip_f000120_t0006.0.jpg'        │
# └─────────────────────────────────────────────────────────────────┘

SHARP_IMAGE  = r'..\data\raw\dashcam_frames\CHANGE_ME.jpg'
BLURRY_IMAGE = r'..\data\raw\dashcam_frames\CHANGE_ME.jpg'

# ── Auto-pick if you haven't set paths yet ───────────────────────
# If both paths above still say CHANGE_ME, this block auto-selects
# the sharpest and blurriest frame from dashcam_frames/
dashcam_dir = RAW_DIR / 'dashcam_frames'
if 'CHANGE_ME' in str(SHARP_IMAGE) and dashcam_dir.exists():
    all_frames = sorted(dashcam_dir.glob('*.jpg'))[:60]
    if all_frames:
        scores = []
        for p in all_frames:
            g = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
            if g is not None:
                scores.append((cv2.Laplacian(g, cv2.CV_64F).var(), p))
        scores.sort()
        BLURRY_IMAGE = scores[0][1]
        SHARP_IMAGE  = scores[-1][1]
        print(f'Auto-selected:')
        print(f'  Sharp  : {Path(SHARP_IMAGE).name}  (var={scores[-1][0]:.0f})')
        print(f'  Blurry : {Path(BLURRY_IMAGE).name}  (var={scores[0][0]:.0f})')
    else:
        print('No frames in dashcam_frames/. Set paths manually above.')
else:
    print('Using manually set paths.')
    print(f'  Sharp  : {SHARP_IMAGE}')
    print(f'  Blurry : {BLURRY_IMAGE}')

In [ ]:
def figure_blur_detection(sharp_path, blurry_path, output='blur_detection.png'):
    """
    2 rows × 3 columns:
      Col 1: original frame
      Col 2: greyscale
      Col 3: Laplacian heatmap (hot colormap)
    Row 1 = sharp (KEPT), Row 2 = blurry (REJECTED)
    """
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    row_info = [
        (sharp_path,  'KEPT',     '#2ecc71'),
        (blurry_path, 'REJECTED', '#e74c3c'),
    ]
    for row, (path, verdict, color) in enumerate(row_info):
        img  = cv2.imread(str(path))
        assert img is not None, f'Cannot load: {path}'
        rgb  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        lap  = cv2.Laplacian(gray, cv2.CV_64F)
        var  = lap.var()

        # Col 0 — Original
        axes[row][0].imshow(rgb)
        axes[row][0].set_title(f'Original Frame', fontsize=11)
        # verdict badge
        axes[row][0].text(0.03, 0.96, verdict,
                          transform=axes[row][0].transAxes,
                          fontsize=13, fontweight='bold', color='white',
                          va='top',
                          bbox=dict(facecolor=color, boxstyle='round,pad=0.3',
                                    alpha=0.9))

        # Col 1 — Greyscale
        axes[row][1].imshow(gray, cmap='gray')
        axes[row][1].set_title('Greyscale', fontsize=11)

        # Col 2 — Laplacian heatmap
        axes[row][2].imshow(np.abs(lap), cmap='hot')
        axes[row][2].set_title(f'Laplacian Response\nVariance = {var:.0f}',
                               fontsize=11)

        # Row label
        axes[row][0].set_ylabel(
            f'Variance = {var:.0f}\n→ {verdict}',
            fontsize=11, fontweight='bold', color=color
        )

        for ax in axes[row]:
            ax.axis('off')

    # Threshold annotation
    fig.text(0.5, 0.01,
             'Decision rule:  Variance > 60  →  KEPT   |   Variance < 60  →  REJECTED',
             ha='center', fontsize=11,
             bbox=dict(facecolor='#f0f0f0', boxstyle='round,pad=0.4'))

    plt.suptitle('Laplacian Operator — Frame Quality Filter',
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    save_figure(fig, output)
    plt.show()


figure_blur_detection(SHARP_IMAGE, BLURRY_IMAGE)

---
## Figure 2 — Perceptual Hash Deduplication
**Slide 2, Right half**

Shows 3 visually similar frames (from the same stretch of road) with their
8×8 DCT hash grids below. Hamming distances between pairs are shown.
Only Frame 1 is kept.

**Where to pick:** `data/raw/dashcam_frames/`  
Pick 3 frames with sequential timestamps (similar filenames) — they will look nearly identical.

In [ ]:
# ┌─────────────────────────────────────────────────────────────────┐
# │  USER INPUT — 3 near-identical frames                           │
# │  Pick frames with sequential timestamps from dashcam_frames/   │
# └─────────────────────────────────────────────────────────────────┘

FRAME_1 = r'..\data\raw\dashcam_frames\CHANGE_ME.jpg'
FRAME_2 = r'..\data\raw\dashcam_frames\CHANGE_ME.jpg'
FRAME_3 = r'..\data\raw\dashcam_frames\CHANGE_ME.jpg'

# ── Auto-pick if not set ─────────────────────────────────────────
dashcam_dir = RAW_DIR / 'dashcam_frames'
if 'CHANGE_ME' in str(FRAME_1) and dashcam_dir.exists():
    all_frames = sorted(dashcam_dir.glob('*.jpg'))
    if len(all_frames) >= 3:
        FRAME_1, FRAME_2, FRAME_3 = all_frames[0], all_frames[1], all_frames[2]
        print('Auto-selected first 3 frames:')
        for f in [FRAME_1, FRAME_2, FRAME_3]:
            print(f'  {Path(f).name}')
        print('\nTip: Replace with 3 frames that look visually similar (close timestamps).')
    else:
        print('Not enough frames found. Set paths manually.')
else:
    print('Using manually set paths.')

In [ ]:
def phash(gray):
    """64-bit perceptual hash: resize → DCT → top-left 8×8 → binarize."""
    small = cv2.resize(gray, (32, 32)).astype(np.float32)
    dct   = cv2.dct(small)[:8, :8]
    return dct > np.median(dct)

def hamming(a, b):
    return int(np.count_nonzero(a != b))


def figure_phash_dedup(frame1, frame2, frame3, output='phash_dedup.png'):
    """
    Top row : 3 original frames
    Bottom row : 8×8 DCT hash grid for each frame
    Annotations: Hamming distances between pairs
    """
    paths  = [frame1, frame2, frame3]
    labels = ['Frame 1\n(KEPT)', 'Frame 2\n(DUPLICATE — REJECTED)', 'Frame 3\n(DUPLICATE — REJECTED)']
    label_colors = ['#2ecc71', '#e74c3c', '#e74c3c']

    imgs   = []
    hashes = []
    for p in paths:
        img = cv2.imread(str(p))
        assert img is not None, f'Cannot load: {p}'
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        imgs.append(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        hashes.append(phash(gray))

    d12 = hamming(hashes[0], hashes[1])
    d23 = hamming(hashes[1], hashes[2])
    d13 = hamming(hashes[0], hashes[2])

    fig, axes = plt.subplots(2, 3, figsize=(13, 8),
                              gridspec_kw={'height_ratios': [3, 1.5]})

    for col in range(3):
        # Top: original frame
        axes[0][col].imshow(imgs[col])
        axes[0][col].set_title(labels[col], fontsize=11,
                                fontweight='bold', color=label_colors[col])
        axes[0][col].axis('off')

        # Bottom: 8×8 hash grid
        axes[1][col].imshow(hashes[col].astype(float), cmap='gray',
                             vmin=0, vmax=1, interpolation='nearest')
        axes[1][col].set_title('DCT pHash (8×8)', fontsize=9)
        axes[1][col].set_xticks(np.arange(-0.5, 8, 1), minor=True)
        axes[1][col].set_yticks(np.arange(-0.5, 8, 1), minor=True)
        axes[1][col].grid(which='minor', color='gray', linewidth=0.5)
        axes[1][col].tick_params(which='both', bottom=False, left=False,
                                  labelbottom=False, labelleft=False)

    # Hamming distance annotations between hash grids
    y = -0.35
    for ax, dist, pair in zip(
        [axes[1][0], axes[1][1], axes[1][2]],
        [f'Hamming(1,2) = {d12}/64', f'Hamming(1,3) = {d13}/64', f'Hamming(2,3) = {d23}/64'],
        ['1↔2', '1↔3', '2↔3']
    ):
        threshold_ok = int(dist.split('=')[1].split('/')[0]) <= 6
        color = '#e74c3c' if threshold_ok else '#2c3e50'
        ax.text(0.5, y, dist + (' → Duplicate!' if threshold_ok else ''),
                transform=ax.transAxes, ha='center', fontsize=10,
                color=color, fontweight='bold')

    fig.text(0.5, 0.01,
             'Rule:  Hamming distance ≤ 6 bits  →  frames are near-duplicates  →  keep only the first',
             ha='center', fontsize=10,
             bbox=dict(facecolor='#f0f0f0', boxstyle='round,pad=0.4'))

    plt.suptitle('Perceptual Hashing (DCT) — Near-Duplicate Frame Removal',
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    save_figure(fig, output)
    plt.show()


figure_phash_dedup(FRAME_1, FRAME_2, FRAME_3)

---
## Figure 3 — Severity Crops (ROI Extraction + Auto-labelling)
**Slide 3, Bottom half**

Shows 3 pothole images (one per severity level) with:
- The full image with bounding box drawn
- The extracted 128×128 crop
- Area ratio value and severity label

**Images are read automatically from your pre-selected folders:**
- `data/raw/potholes/small/`  → Low severity
- `data/raw/potholes/medium/` → Medium severity
- `data/raw/potholes/large/`  → High severity

Each folder contains `.jpg` images with matching YOLO `.txt` label files  
(`class_id x_center y_center width height`, all values normalised 0–1).

To use a specific image instead of the auto-picked one, set the override path in the USER INPUT cell.

In [ ]:
# ── Quick kernel check — run this first, should complete in < 5 seconds ──
import sys, time
print('Python:', sys.executable)
print('Checking torch...', end=' ', flush=True)
t = time.time()
import torch
print(f'OK  ({time.time()-t:.1f}s)')
print('Checking ultralytics...', end=' ', flush=True)
t = time.time()
from ultralytics import YOLO
print(f'OK  ({time.time()-t:.1f}s)')
from pathlib import Path
model_path = Path('..').resolve() / 'models/severity_model.pt'
print('Model file exists:', model_path.exists(), f'({model_path.stat().st_size/1e6:.1f} MB)' if model_path.exists() else '')
print('Loading model...', end=' ', flush=True)
t = time.time()
m = YOLO(str(model_path))
print(f'OK  ({time.time()-t:.1f}s)')
print('All good — run the figure cell below.')

In [ ]:
# ── STANDALONE: Severity model inference on 3 pothole images ──────────────
# Run this single cell. No other cells needed.

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from ultralytics import YOLO

BASE  = Path('..').resolve()
MODEL = YOLO(str(BASE / 'models/severity_model.pt'))

# YOLO-cls sorts classes alphabetically → High=0, Low=1, Medium=2
IDX_TO_CLASS = {0: 'High', 1: 'Low', 2: 'Medium'}
SEV_COLORS   = {'Low': '#27ae60', 'Medium': '#f39c12', 'High': '#e74c3c'}
PADDING      = 10

IMAGES = [
    (BASE / 'data/raw/potholes/small/1067_1_img-273_jpg.rf.5cb7aacf7db557524549b408bc043e8b.jpg',  'Low'),
    (BASE / 'data/raw/potholes/medium/779_5_img-74_jpg.rf.47c99578f55c2761e6d4f8bcf021d6f2.jpg',   'Medium'),
    (BASE / 'data/raw/potholes/large/128_5_img-541_jpg.rf.ef924ac7c487acc572f57d8daf390a49.jpg',   'High'),
]

def read_yolo_box(txt_path, img_w, img_h):
    for line in Path(txt_path).read_text().strip().splitlines():
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        xc, yc, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
        return (
            max(int((xc - bw/2) * img_w), 0),
            max(int((yc - bh/2) * img_h), 0),
            min(int((xc + bw/2) * img_w), img_w),
            min(int((yc + bh/2) * img_h), img_h),
        )
    return None

fig, axes = plt.subplots(2, 3, figsize=(13, 8),
                          gridspec_kw={'height_ratios': [3, 1.5]})

for col, (img_path, true_label) in enumerate(IMAGES):
    txt_path = img_path.with_suffix('.txt')
    img = cv2.imread(str(img_path))
    assert img is not None, f'Cannot load: {img_path}'
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    box = read_yolo_box(txt_path, w, h)
    assert box is not None, f'No annotation in: {txt_path}'
    xmin, ymin, xmax, ymax = box

    # Crop → resize to 128×128 → run model
    x1, y1 = max(xmin - PADDING, 0), max(ymin - PADDING, 0)
    x2, y2 = min(xmax + PADDING, w), min(ymax + PADDING, h)
    crop_bgr  = cv2.resize(img[y1:y2, x1:x2], (128, 128))
    result    = MODEL(crop_bgr, imgsz=128, verbose=False)[0]
    pred_idx  = int(result.probs.top1)
    pred_cls  = IDX_TO_CLASS[pred_idx]
    conf      = float(result.probs.top1conf)

    color = SEV_COLORS[pred_cls]

    # Row 0 — full image + bounding box
    axes[0][col].imshow(rgb)
    axes[0][col].add_patch(patches.Rectangle(
        (xmin, ymin), xmax - xmin, ymax - ymin,
        linewidth=3, edgecolor=color, facecolor='none'))
    axes[0][col].set_title(f'Model prediction: {pred_cls}\nConf: {conf:.2%}',
                            fontsize=12, fontweight='bold', color=color)
    axes[0][col].axis('off')

    # Row 1 — 128×128 crop fed to model
    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    axes[1][col].imshow(crop_rgb)
    axes[1][col].set_title('Input crop (128×128)', fontsize=9)
    axes[1][col].axis('off')
    axes[1][col].text(0.5, -0.12,
                      f'Predicted: {pred_cls}  ({conf:.1%})',
                      transform=axes[1][col].transAxes,
                      ha='center', fontsize=11, fontweight='bold', color=color)

plt.suptitle('Severity Classification Model — Inference Results',
             fontsize=14, fontweight='bold')
plt.tight_layout()

out = BASE / 'severity_crops.png'
fig.savefig(str(out), dpi=150, bbox_inches='tight', facecolor='white')
print(f'Saved → {out}')
plt.show()

---
## Figure 4 — Weather Augmentation Grid (5 × 5)
**Slide 4 — The showstopper slide**

5 columns = 5 images you choose.  
5 rows = Original / Rain / Fog / Night / Wet Road.

**Where to pick:** Any 5 images from `data/raw/pothole/` or `data/raw/dashcam_frames/`  
Pick images with variety — some with potholes, some road-only, different lighting.

In [ ]:
# ┌─────────────────────────────────────────────────────────────────────────┐
# │  USER INPUT — pick 5 images for the augmentation grid                  │
# │  Mix of pothole images and dashcam frames works well                   │
# └─────────────────────────────────────────────────────────────────────────┘

AUG_IMAGE_1 = r'CHANGE_ME.jpg'
AUG_IMAGE_2 = r'CHANGE_ME.jpg'
AUG_IMAGE_3 = r'CHANGE_ME.jpg'
AUG_IMAGE_4 = r'CHANGE_ME.jpg'
AUG_IMAGE_5 = r'CHANGE_ME.jpg'

# ── Auto-pick if not set ─────────────────────────────────────────────────
if 'CHANGE_ME' in str(AUG_IMAGE_1):
    # Try dashcam_frames first, then chitholian
    pool = []
    for folder in [RAW_DIR/'dashcam_frames',
                   RAW_DIR/'pothole'/'annotated-pothole-images-chitholian']:
        if folder.exists():
            pool += sorted(folder.glob('*.jpg'))
    if len(pool) >= 5:
        # spread picks across the pool
        step = max(1, len(pool) // 5)
        picks = [pool[i * step] for i in range(5)]
        AUG_IMAGE_1, AUG_IMAGE_2, AUG_IMAGE_3, AUG_IMAGE_4, AUG_IMAGE_5 = picks
        print('Auto-selected 5 images:')
        for p in picks:
            print(f'  {p.name}')
    else:
        print('Not enough images found. Set paths manually above.')
else:
    print('Using manually set paths.')

AUG_IMAGES = [AUG_IMAGE_1, AUG_IMAGE_2, AUG_IMAGE_3, AUG_IMAGE_4, AUG_IMAGE_5]

In [ ]:
import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TF
import random

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Augmentation device:', DEVICE)

def to_tensor(img_bgr):
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    return torch.from_numpy(rgb).permute(2,0,1).float().div(255.0).to(DEVICE)

def to_img(t):
    rgb = t.clamp(0,1).mul(255).byte().permute(1,2,0).cpu().numpy()
    return cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)

def aug_rain(t):
    C, H, W = t.shape
    rain = torch.zeros(1, H, W, device=DEVICE)
    xs = torch.randint(0, W, (700,), device=DEVICE)
    ys = torch.randint(0, H, (700,), device=DEVICE)
    rain[0, ys, xs] = torch.FloatTensor(700).uniform_(0.6, 1.0).to(DEVICE)
    L = 16
    kernel = torch.zeros(1,1,L,1, device=DEVICE)
    kernel[0,0,:,0] = 1.0/L
    rain = F.conv2d(rain.unsqueeze(0), kernel, padding=(L//2,0)).squeeze(0)[:, :H, :]
    return torch.clamp(t * 0.88 + rain.expand(C,-1,-1) * 0.55, 0, 1)

def aug_fog(t):
    i = random.uniform(0.40, 0.52)
    return torch.clamp(t * (1.0 - i) + i, 0, 1)

def aug_night(t):
    out = t * random.uniform(0.20, 0.28)
    out[2] = torch.clamp(out[2] + 0.04, 0, 1)
    return out

def aug_wet(t):
    c = t.cpu()
    c = TF.adjust_saturation(c, random.uniform(1.2, 1.5))
    c = TF.adjust_brightness(c, random.uniform(0.75, 0.90))
    return c.to(DEVICE)


def figure_augmentation_grid(image_paths, output='augmentation_grid.png'):
    """
    5 rows (Original / Rain / Fog / Night / Wet Road) ×
    N columns (one per input image)
    """
    random.seed(42)
    torch.manual_seed(42)

    effects = [
        ('Original',   lambda t: t,       '#2c3e50'),
        ('Rain',       aug_rain,           '#3498db'),
        ('Fog',        aug_fog,            '#95a5a6'),
        ('Night',      aug_night,          '#8e44ad'),
        ('Wet Road',   aug_wet,            '#1abc9c'),
    ]

    n = len(image_paths)
    fig, axes = plt.subplots(5, n, figsize=(3.2*n, 14))

    for col, p in enumerate(image_paths):
        img = cv2.imread(str(p))
        assert img is not None, f'Cannot load: {p}'
        # Resize for consistent display
        img = cv2.resize(img, (480, 320))
        t   = to_tensor(img)

        for row, (label, fn, label_color) in enumerate(effects):
            out = to_img(fn(t.clone()))
            axes[row][col].imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB))
            axes[row][col].axis('off')

            # Row label on leftmost column only
            if col == 0:
                axes[row][col].set_ylabel(
                    label, fontsize=13, fontweight='bold',
                    color=label_color, rotation=0,
                    labelpad=65, va='center'
                )

            # IP technique annotation under each row label
            if col == 0:
                subtitles = ['', '1D Conv Kernel', 'Alpha Blending',
                             'Gamma Correction', 'HSV Colour Space']
                if subtitles[row]:
                    axes[row][col].text(
                        -0.42, 0.5, subtitles[row],
                        transform=axes[row][col].transAxes,
                        fontsize=8, color='gray', va='center',
                        rotation=0, style='italic'
                    )

    plt.suptitle('Weather Augmentation — 4 Image Processing Techniques',
                 fontsize=15, fontweight='bold')
    plt.tight_layout()
    save_figure(fig, output)
    plt.show()


figure_augmentation_grid(AUG_IMAGES)

---
## Quick Re-run — Generate All Figures at Once

After you have set all paths above and confirmed each figure looks good,
run this cell to regenerate all 4 figures in one shot.

In [ ]:
print('Generating all figures...')
print()

print('[1/4] Blur detection...')
figure_blur_detection(SHARP_IMAGE, BLURRY_IMAGE)

print('[2/4] pHash deduplication...')
figure_phash_dedup(FRAME_1, FRAME_2, FRAME_3)

print('[3/4] Severity crops...')
figure_severity_crops(low_img, low_txt, med_img, med_txt, high_img, high_txt)

print('[4/4] Augmentation grid...')
figure_augmentation_grid(AUG_IMAGES)

print()
print('All figures saved to project root:')
for f in ['blur_detection.png', 'phash_dedup.png',
           'severity_crops.png', 'augmentation_grid.png']:
    p = OUTPUT_DIR / f
    status = 'OK' if p.exists() else 'MISSING'
    print(f'  [{status}]  {f}')